# NCES Private School Data Download

### Importing necessary libraries

In [22]:
import time, os
import pandas as pd

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

### Setting up state mappings, columns to note, and chrome options for Selenium driver

In [23]:
# FIPS code for each State and Territory to use in URL construction for downloading school data for each state
state_FIPS = {
    'Alabama': '01',
    'Alaska': '02',
    'Arizona': '04',
    'Arkansas': '05',
    'California': '06',
    'Colorado': '08',
    'Connecticut': '09',
    'Delaware': '10',
    'Florida': '12',
    'Georgia': '13',
    'Hawaii': '15',
    'Idaho': '16',
    'Illinois': '17',
    'Indiana': '18',
    'Iowa': '19',
    'Kansas': '20',
    'Kentucky': '21',
    'Louisiana': '22',
    'Maine': '23',
    'Maryland': '24',
    'Massachusetts': '25',
    'Michigan': '26',
    'Minnesota': '27',
    'Mississippi': '28',
    'Missouri': '29',
    'Montana': '30',
    'Nebraska': '31',
    'Nevada': '32',
    'New Hampshire': '33',
    'New Jersey': '34',
    'New Mexico': '35',
    'New York': '36',
    'North Carolina': '37',
    'North Dakota': '38',
    'Ohio': '39',
    'Oklahoma': '40',
    'Oregon': '41',
    'Pennsylvania': '42',
    'Rhode Island': '44',
    'South Carolina': '45',
    'South Dakota': '46',
    'Tennessee': '47',
    'Texas': '48',
    'Utah': '49',
    'Vermont': '50',
    'Virginia': '51',
    'Washington': '53',
    'West Virginia': '54',
    'Wisconsin': '55',
    'Wyoming': '56'
}

keep_cols = ['PSS_SCHOOL_ID', 'PSS_INST', 'LoGrade', 'HiGrade', 'PSS_ADDRESS', 'PSS_CITY', 'PSS_STABB', 'PSS_ZIP5', 'PSS_PHONE',
             'PSS_ENROLL_T', 'PSS_FTE_TEACH', 'PSS_RELIG', 'PSS_COMM_TYPE', 'PSS_COUNTY_NAME', 'PSS_ASSOC_1', 'PSS_ASSOC_2', 'PSS_ASSOC_3']

final_cols = ['NCES School ID', 'Account Name', 'Low Grade', 'High Grade', 'Billing Street', 'Billing City', 'Billing State', 'Billing ZIP',
              'Phone', 'Number of Students Served', 'Number of Teachers', 'School Type', 'School Environment', 'County Name']

# Set download directory
download_dir = f'{os.getcwd()}\\priv_downloads'
os.makedirs(download_dir, exist_ok=True) # Ensure the download directory exists

# Configure Chrome options for automatic download
chrome_options = webdriver.ChromeOptions()
prefs = {"download.default_directory": download_dir} # Establish and add download directory preference
chrome_options.add_experimental_option("prefs", prefs)
chrome_options.add_argument("--headless=new")  # Run Chrome in headless mode (optional)

### Function to Standardize Columns Kept

In [ ]:
def create_sd_csv(excel_file, state):
    df = pd.concat(excel_file, ignore_index=True)

    # Change column names to values in keep_cols
    df.columns = df.loc[4]
    df = df[keep_cols]
    df = df.loc[5:].reset_index(drop=True) # Get only rows containing school data, reset index

    # Create masks to handle empty values for different columns
    num_students_empty_mask = (df['PSS_ENROLL_T'] == '–') | (df['PSS_ENROLL_T'] == '†') | (df['PSS_ENROLL_T'].isna())
    num_teachers_empty_mask = (df['PSS_FTE_TEACH'] == '–') | (df['PSS_FTE_TEACH'] == '†') | (df['PSS_FTE_TEACH'].isna())
    ungraded_grade_empty_mask = (df['LoGrade'] == '–') & (df['HiGrade'] == '–')

    # Assign empty string to empty values, convert non-empty values to appropriate data types
    df.loc[num_students_empty_mask, 'PSS_ENROLL_T'] = ''
    df.loc[~num_students_empty_mask, 'PSS_ENROLL_T'] = df.loc[~num_students_empty_mask, 'PSS_ENROLL_T'].astype(int).astype(str)

    df.loc[num_teachers_empty_mask, 'PSS_FTE_TEACH'] = ''
    df.loc[~num_teachers_empty_mask, 'PSS_FTE_TEACH'] = df.loc[~num_teachers_empty_mask, 'PSS_FTE_TEACH'].astype(float).astype(str)

    # Create Locale mappings/masks and update Locale column based on Locale Code values
    urban_mask = (df['PSS_COMM_TYPE'] == '1')
    suburban_mask = df['PSS_COMM_TYPE'].isin(['2','3'])
    rural_mask = (df['PSS_COMM_TYPE'] == '4')

    df.loc[urban_mask, 'PSS_COMM_TYPE'] = 'Urban'
    df.loc[suburban_mask, 'PSS_COMM_TYPE'] = 'Suburban'
    df.loc[rural_mask, 'PSS_COMM_TYPE'] = 'Rural'

    # Calculating Low and High Grade Levels based on encoded values in columns
    grade_map = {'-1': '', '1': '', '2': 'PK', '3': 'KG', '4': 'KG', '5': 'KG'} # Map for ungraded schools
    # print(list(grade_map.keys()))

    low_grade_kg_and_lower_mask = (df['LoGrade'].isin(list(grade_map.keys())))
    high_grade_kg_and_lower_mask = (df['HiGrade'].isin(list(grade_map.keys())))

    df.loc[low_grade_kg_and_lower_mask, 'LoGrade'] = df.loc[low_grade_kg_and_lower_mask, 'LoGrade'].map(grade_map)
    df.loc[high_grade_kg_and_lower_mask, 'HiGrade'] = df.loc[high_grade_kg_and_lower_mask, 'HiGrade'].map(grade_map)

    df.loc[~low_grade_kg_and_lower_mask, 'LoGrade'] = (df.loc[~low_grade_kg_and_lower_mask, 'LoGrade'].astype(int) - 5).astype(str)
    df.loc[~high_grade_kg_and_lower_mask, 'HiGrade'] = (df.loc[~high_grade_kg_and_lower_mask, 'HiGrade'].astype(int) - 5).astype(str)

    # Fix phone number formatting
    phone_empty_mask = (df['PSS_PHONE'] == '–') | (df['PSS_PHONE'] == '†') | (df['PSS_PHONE'].isna())
    phone_valid_mask = ~phone_empty_mask & df['PSS_PHONE'].str.match(r'^[2-9]\d{9}$') # Some phone numbers are invalid, so they are temporarily set to empty values to be evaluated later

    df.loc[phone_empty_mask | ~phone_valid_mask, 'PSS_PHONE'] = ''
    df.loc[phone_valid_mask, 'PSS_PHONE'] = '(' + df.loc[phone_valid_mask, 'PSS_PHONE'].str.slice(0, 3) + ') ' + df.loc[phone_valid_mask, 'PSS_PHONE'].str.slice(3, 6) + '-' + df.loc[phone_valid_mask, 'PSS_PHONE'].str.slice(6)

    # Change state abbreviation to full name
    df['PSS_STABB'] = state

    # Determine school types based on Religious/Indepenent affiliate/association status
    religious_mask = (df['PSS_RELIG'].isin(['1','2']))
    independent_mask = (df['PSS_ASSOC_1'].str.lower().str.contains('independent', na=False)) | (df['PSS_ASSOC_2'].str.lower().str.contains('independent', na=False)) | (df['PSS_ASSOC_3'].str.lower().str.contains('independent', na=False))

    df.loc[religious_mask, 'PSS_RELIG'] = 'Parochial'
    df.loc[independent_mask, 'PSS_RELIG'] = 'Independent'
    df.loc[~religious_mask & ~independent_mask, 'PSS_RELIG'] = 'Private'

    df.drop(columns=['PSS_ASSOC_1', 'PSS_ASSOC_2', 'PSS_ASSOC_3'], inplace=True) # Drop association columns since they won't be used in final version

    # Update columns to final version we want
    df.columns = final_cols

    # Add 'School' type label for each entry
    df['Type'] = 'School'

    # Temp output
    df.to_csv(f'{download_dir}\\{state}_sd.csv', index=False)

### Main Loop to Retrieve, Standardize, and Save Files

In [ ]:
# Logic to navigate to School Data Excel file and download it
for state, fips in state_FIPS.items():

    # Create fresh Selenium driver instance
    driver = webdriver.Chrome(options=chrome_options)

    # Construct URL for each state using FIPS code
    url = f"""https://nces.ed.gov/surveys/pss/privateschoolsearch/school_list.asp?Search=1&SchoolName=&SchoolID=&Address=&City=&State={fips}&Zip=&Miles=&County=&PhoneAreaCode=&Phone=&Religion=&Association=&SchoolType=&Coed=&NumOfStudents=&NumOfStudentsRange=more&IncGrade=-1&LoGrade=-1&HiGrade=-1"""

    print(f'Downloading private school data for {state}...')
    try:
        files_before_download = set(os.listdir(download_dir)) # Snapshot directory before download so we can identify exactly which file this state produces

        driver.get(url) # Navigate to the specified URL
        wait = WebDriverWait(driver, 10) # Initialize WebDriverWait with a timeout of 10 seconds

        # print(f'Before clicking Excel link: {len(driver.window_handles)}') # For Debugging: Check number of windows before clicking Excel link

        # Click on link that opens window with Excel download link
        wait.until(EC.element_to_be_clickable((By.CLASS_NAME, 'excelclass'))).click()
        time.sleep(20)

        # print(f'After clicking Excel link: {len(driver.window_handles)}') # For Debugging: Check number of windows after clicking Excel link to ensure new window opened

        # Switch to the new window that opened after clicking the Excel link
        driver.switch_to.window(driver.window_handles[-1])

        wait.until(EC.element_to_be_clickable((By.LINK_TEXT, 'Download Excel File'))).click() # Click Excel download link in new window

        # Poll for the download to complete instead of a fixed sleep
        download_timeout = 30 # seconds to wait for the download to complete
        poll_interval = 0.5
        elapsed = 0
        new_files = []
        while elapsed < download_timeout:
            new_files = [f for f in os.listdir(download_dir) if f not in files_before_download and (f.endswith('.xlsx') or f.endswith('.xls'))]
            if new_files:
                break
            time.sleep(poll_interval)
            elapsed += poll_interval

        if new_files:
            downloaded_file = os.path.join(download_dir, new_files[0])
            try:
                excel_file = pd.read_html(downloaded_file) # Read the downloaded Excel file into a DataFrame

                print(f'{state}: {new_files[0]}')
                create_sd_csv(excel_file, state) # Create CSV file from the downloaded Excel file
            finally:
                os.remove(downloaded_file) # Always remove the original Excel file, even if parsing failed
    except Exception as e:
        print(f'Error downloading data for {state}: {e}')
    finally:
        print(f'{state} Download Complete\n')
        driver.quit() # Hard reset driver after each state

### Testing Cell

In [27]:
test_states = ['Alabama', 'Alaska', 'Arizona'] # First three states alphabetically, for testing

for state in test_states:
    fips = state_FIPS[state]

    # Create fresh Selenium driver instance
    driver = webdriver.Chrome(options=chrome_options)

    # Construct URL for each state using FIPS code
    url = f"""https://nces.ed.gov/surveys/pss/privateschoolsearch/school_list.asp?Search=1&SchoolName=&SchoolID=&Address=&City=&State={fips}&Zip=&Miles=&County=&PhoneAreaCode=&Phone=&Religion=&Association=&SchoolType=&Coed=&NumOfStudents=&NumOfStudentsRange=more&IncGrade=-1&LoGrade=-1&HiGrade=-1"""

    print(f'Downloading private school data for {state}...')
    try:
        files_before_download = set(os.listdir(download_dir)) # Snapshot directory before download so we can identify exactly which file this state produces

        driver.get(url) # Navigate to the specified URL
        wait = WebDriverWait(driver, 10) # Initialize WebDriverWait with a timeout of 10 seconds

        # print(f'Before clicking Excel link: {len(driver.window_handles)}') # For Debugging: Check number of windows before clicking Excel link

        # Click on link that opens window with Excel download link
        wait.until(EC.element_to_be_clickable((By.CLASS_NAME, 'excelclass'))).click()
        time.sleep(20)

        # print(f'After clicking Excel link: {len(driver.window_handles)}') # For Debugging: Check number of windows after clicking Excel link to ensure new window opened

        # Switch to the new window that opened after clicking the Excel link
        driver.switch_to.window(driver.window_handles[-1])

        wait.until(EC.element_to_be_clickable((By.LINK_TEXT, 'Download Excel File'))).click() # Click Excel download link in new window

        # Poll for the download to complete instead of a fixed sleep
        download_timeout = 30 # seconds to wait for the download to complete
        poll_interval = 0.5
        elapsed = 0
        new_files = []
        while elapsed < download_timeout:
            new_files = [f for f in os.listdir(download_dir) if f not in files_before_download and (f.endswith('.xlsx') or f.endswith('.xls'))]
            if new_files:
                break
            time.sleep(poll_interval)
            elapsed += poll_interval

        if new_files:
            downloaded_file = os.path.join(download_dir, new_files[0])
            try:
                excel_file = pd.read_html(downloaded_file) # Read the downloaded Excel file into a DataFrame

                print(f'{state}: {new_files[0]}')
                create_sd_csv(excel_file, state) # Create CSV file from the downloaded Excel file
            finally:
                os.remove(downloaded_file) # Always remove the original Excel file, even if parsing failed
    except Exception as e:
        print(f'Error downloading data for {state}: {e}')
    finally:
        print(f'{state} Download Complete\n')
        driver.quit() # Hard reset driver after each state

Alabama: ncesdata_860EC268.xls
Alabama Download Complete

Alaska: ncesdata_97D1A903.xls
Alaska Download Complete

Arizona: ncesdata_3BE6CE44.xls
Arizona Download Complete

